### Cohort filtering (for 1 <= LOS <= 25, hospital_expire_flag == 0, age >= 18)
Merging necessary columns from **admissions** and **patients**

In [1]:
import pandas as pd

ind_hosp = pd.read_csv('mimic-iv-3.1/hosp/admissions.csv.gz')
ind_hosp = ind_hosp.drop(columns=['deathtime', 'admit_provider_id', 'language', 'marital_status', 'edregtime', 'edouttime'])

ind_hosp['admittime'] = pd.to_datetime(ind_hosp['admittime'])
ind_hosp['dischtime'] = pd.to_datetime(ind_hosp['dischtime'])
ind_hosp['los'] = (ind_hosp['dischtime'] - ind_hosp['admittime']).dt.days

ind_hosp = ind_hosp[(ind_hosp['los'] >= 1) & (ind_hosp['los'] <= 25)
                     & (ind_hosp['hospital_expire_flag'] == 0)]
ind_hosp = ind_hosp.drop(columns=['hospital_expire_flag'])

In [2]:
patients = pd.read_csv('mimic-iv-3.1/hosp/patients.csv.gz').drop(columns=['dod', 'anchor_year_group'])

ind_hosp = ind_hosp.merge(patients, on='subject_id', how='left')
ind_hosp['age'] = ind_hosp['anchor_age'] + (ind_hosp['admittime'].dt.year - ind_hosp['anchor_year'])
ind_hosp = ind_hosp.drop(columns=['anchor_age', 'anchor_year'])

ind_hosp = ind_hosp[(ind_hosp['age'] >= 18)]

In [3]:
max_days = 25
ind_hosp['days_list'] = ind_hosp['los'].apply(lambda x: list(range(1, min(x, max_days) + 1)))
expanded = ind_hosp.explode('days_list').reset_index(drop=True)
expanded = expanded.rename(columns={'days_list': 'day'})
expanded['current_date'] = expanded['admittime'] + pd.to_timedelta(expanded['day'] - 1, unit='D')
expanded['current_date'] = expanded['current_date'].dt.date

if 'days_list' in expanded.columns:
    expanded = expanded.drop(columns=['days_list'])

ind_hosp = expanded
ind_hosp

,subject_id,hadm_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,race,los,gender,age,day,current_date
0,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,1,F,52,1,2180-06-26
1,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.,EMERGENCY ROOM,HOSPICE,Medicaid,WHITE,1,F,52,1,2180-08-05
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,2,F,52,1,2180-07-23
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,2,F,52,2,2180-07-24
4,10000084,23052089,2160-11-21 01:56:00,2160-11-25 14:52:00,EW EMER.,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,WHITE,4,M,72,1,2160-11-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1825053,19999987,23865745,2145-11-02 21:38:00,2145-11-11 12:57:00,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,F,57,4,2145-11-05
1825054,19999987,23865745,2145-11-02 21:38:00,2145-11-11 12:57:00,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,F,57,5,2145-11-06
1825055,19999987,23865745,2145-11-02 21:38:00,2145-11-11 12:57:00,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,F,57,6,2145-11-07
1825056,19999987,23865745,2145-11-02 21:38:00,2145-11-11 12:57:00,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,F,57,7,2145-11-08


In [4]:
print(f"Expanded size: {len(ind_hosp)} rows")
print(f"Unique readmissions: {ind_hosp['hadm_id'].nunique()}")
print(f"LOS range: {ind_hosp['day'].min()} - {ind_hosp['day'].max()}")

Expanded size: 1825058 rows
Unique readmissions: 406285
LOS range: 1 - 25


In [5]:
ind_hosp.isna().sum()

subject_id                0
hadm_id                   0
admittime                 0
dischtime                 0
admission_type            0
admission_location        0
discharge_location    79737
insurance             19694
race                      0
los                       0
gender                    0
age                       0
day                       0
current_date              0
dtype: int64

### Diagnoses mapping to ICD-10, calculation of Charlson, CCSR, Chronic Condition Indicator

In [6]:
from icdmappings import Mapper

diagnoses_icd = pd.read_csv('mimic-iv-3.1/hosp/diagnoses_icd.csv.gz')

mapper = Mapper()
diagnoses_icd['icd10_code'] = diagnoses_icd.apply(
    lambda row: mapper.map(row['icd_code'], source='icd9', target='icd10') 
                if row['icd_version'] == 9 
                else row['icd_code'],
    axis=1
)
diagnoses_icd = diagnoses_icd.drop(columns=['icd_code', 'icd_version'])
diagnoses_icd = diagnoses_icd.dropna(subset=['icd10_code'])

diagnoses_icd['ccsr_category'] = diagnoses_icd['icd10_code'].apply(
    lambda code: mapper.map(code, source='icd10', target='ccsr')
)
diagnoses_icd['is_chronic'] = diagnoses_icd['icd10_code'].apply(
    lambda code: mapper.map(code, source='icd10', target='ccir')
)
diagnoses_icd = diagnoses_icd.merge(ind_hosp[['age', 'hadm_id']], on='hadm_id', how='inner')

diagnoses_icd

,subject_id,hadm_id,seq_num,icd10_code,ccsr_category,is_chronic,age
0,10000032,22841357,1,B1921,INF007,True,52
1,10000032,22841357,2,R188,SYM006,False,52
2,10000032,22841357,3,D696,BLD006,False,52
3,10000032,22841357,4,E871,END011,False,52
4,10000032,22841357,5,J449,RSP008,True,52
...,...,...,...,...,...,...,...
28446319,19999987,23865745,11,R259,SYM010,False,57
28446320,19999987,23865745,11,R259,SYM010,False,57
28446321,19999987,23865745,11,R259,SYM010,False,57
28446322,19999987,23865745,11,R259,SYM010,False,57


In [7]:
diag_summary = diagnoses_icd.groupby('hadm_id').agg(
    num_diagnoses=('hadm_id', 'size'),
    num_chronic=('is_chronic', 'sum')
).reset_index()

top15_ccsr = diagnoses_icd['ccsr_category'].value_counts().head(15).index.tolist()
print(f"Top-15 CCSR: {top15_ccsr}")

for cat in top15_ccsr:
    if pd.notna(cat):
        hadm_with_cat = diagnoses_icd[diagnoses_icd['ccsr_category'] == cat]['hadm_id'].unique()
        diag_summary[f'ccsr_{cat}'] = diag_summary['hadm_id'].isin(hadm_with_cat).astype(int)

Top-15 CCSR: ['FAC021', 'FAC025', 'END011', 'CIR011', 'END010', 'CIR007', 'END003', 'CIR019', 'DIG004', 'CIR017', 'CIR008', 'BLD003', 'EXT027', 'GEN002', 'END009']


In [8]:
from comorbidipy import comorbidity
import polars as pl

diagnoses_icd = pl.DataFrame(diagnoses_icd)
result = comorbidity(diagnoses_icd, id_col="hadm_id", code_col="icd10_code", age_col="age")
result

hadm_id,age,rend,diabwc,aids,hp,rheumd,msld,cevd,pud,canc,diab,mld,chf,copd,pvd,metacanc,ami,dementia,comorbidity_score
i64,i64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64
22318231,36,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0
27775219,72,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0
21718385,24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0
22623766,54,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,2.0
20847481,35,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,4.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
28354792,32,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0
22737629,47,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0
20748465,73,1,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,4.0


In [9]:
top15_icd = diagnoses_icd['icd10_code'].value_counts().sort('count', descending=True).head(15)['icd10_code'].to_list()
print(f"Top-15 ICD: {top15_icd}")

for code in top15_icd:
    col_name = f'icd_{code.replace(".", "_")}'
    hadm_with_code = diagnoses_icd.filter(pl.col('icd10_code') == code)['hadm_id'].unique().to_list()
    diag_summary[col_name] = diag_summary['hadm_id'].isin(hadm_with_code).astype(int)

Top-15 ICD: ['I10', 'E785', 'K219', 'Z87891', 'I2510', 'N179', 'F329', 'I4891', 'Z7901', 'F419', 'E119', 'E039', 'Z794', 'D649', 'N390']


In [10]:
result = result.to_pandas()
diagnosis_features = diag_summary.merge(
    result[['hadm_id', 'comorbidity_score']], 
    on='hadm_id', 
    how='left'
)
print(diagnosis_features.columns)
diagnosis_features

Index(['hadm_id', 'num_diagnoses', 'num_chronic', 'ccsr_FAC021', 'ccsr_FAC025',
       'ccsr_END011', 'ccsr_CIR011', 'ccsr_END010', 'ccsr_CIR007',
       'ccsr_END003', 'ccsr_CIR019', 'ccsr_DIG004', 'ccsr_CIR017',
       'ccsr_CIR008', 'ccsr_BLD003', 'ccsr_EXT027', 'ccsr_GEN002',
       'ccsr_END009', 'icd_I10', 'icd_E785', 'icd_K219', 'icd_Z87891',
       'icd_I2510', 'icd_N179', 'icd_F329', 'icd_I4891', 'icd_Z7901',
       'icd_F419', 'icd_E119', 'icd_E039', 'icd_Z794', 'icd_D649', 'icd_N390',
       'comorbidity_score'],
      dtype='object')


,hadm_id,num_diagnoses,num_chronic,ccsr_FAC021,ccsr_FAC025,ccsr_END011,ccsr_CIR011,ccsr_END010,ccsr_CIR007,ccsr_END003,...,icd_F329,icd_I4891,icd_Z7901,icd_F419,icd_E119,icd_E039,icd_Z794,icd_D649,icd_N390,comorbidity_score
0,20000019,24,12,1,0,1,0,1,1,0,...,0,0,0,0,1,0,0,1,0,1.0
1,20000034,28,11,1,1,1,0,0,0,1,...,0,0,0,0,0,0,1,0,0,5.0
2,20000041,30,18,1,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0.0
3,20000045,112,63,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,6.0
4,20000057,46,24,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,1,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405974,29999745,42,18,1,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0.0
405975,29999803,390,182,1,0,1,0,1,0,0,...,1,0,0,1,0,1,0,1,0,4.0
405976,29999809,75,50,1,0,0,1,1,1,0,...,0,0,0,0,1,0,0,0,1,1.0
405977,29999828,24,12,0,0,1,0,1,1,0,...,0,0,0,0,1,0,0,0,0,0.0


In [11]:
ind_hosp = ind_hosp.merge(diagnosis_features, on="hadm_id", how='left')

### Procedures preprocessing (number of procedures daily + cumulative)

In [12]:
procedures_icd = pd.read_csv('mimic-iv-3.1/hosp/procedures_icd.csv.gz')

procedures_icd['chartdate'] = pd.to_datetime(procedures_icd['chartdate'])
procedures_daily = procedures_icd.groupby(['hadm_id', 'chartdate']).size().reset_index(name='proc_count_daily')

proc_dict = {}
for _, row in procedures_daily.iterrows():
    key = (row['hadm_id'], row['chartdate'].date())
    proc_dict[key] = row['proc_count_daily']

ind_hosp['proc_key'] = list(zip(ind_hosp['hadm_id'], ind_hosp['current_date']))
ind_hosp['num_procedures_daily'] = ind_hosp['proc_key'].map(proc_dict).fillna(0).astype(int)
ind_hosp = ind_hosp.drop(columns=['proc_key'])

ind_hosp = ind_hosp.sort_values(['subject_id', 'hadm_id', 'day'])
ind_hosp['cumulative_procedures'] = ind_hosp.groupby('hadm_id')['num_procedures_daily'].cumsum()

### Prescriptions preprocessing (number of MAIN drugs daily + cumulative)

In [13]:
prescriptions = pd.read_csv('mimic-iv-3.1/hosp/prescriptions.csv.gz')
prescriptions

/var/folders/m0/9dr3jfm12bj1fkg2xjqh62jc0000gn/T/ipykernel_7037/3577613178.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  prescriptions = pd.read_csv('mimic-iv-3.1/hosp/prescriptions.csv.gz')


,subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,starttime,stoptime,drug_type,drug,...,gsn,ndc,prod_strength,form_rx,dose_val_rx,dose_unit_rx,form_val_disp,form_unit_disp,doses_per_24_hrs,route
0,10000032,22595853,12775705,10000032-55,55.0,P85UQ1,2180-05-08 08:00:00,2180-05-07 22:00:00,MAIN,Furosemide,...,008209,5.107901e+10,40mg Tablet,NaN,40,mg,1,TAB,1.0,PO/NG
1,10000032,22595853,18415984,10000032-42,42.0,P23SJA,2180-05-07 02:00:00,2180-05-07 22:00:00,MAIN,Ipratropium Bromide Neb,...,021700,4.879801e+08,2.5mL Vial,NaN,1,NEB,1,VIAL,4.0,IH
2,10000032,22595853,23637373,10000032-35,35.0,P23SJA,2180-05-07 01:00:00,2180-05-07 09:00:00,MAIN,Furosemide,...,008208,5.107901e+10,20mg Tablet,NaN,20,mg,1,TAB,1.0,PO/NG
3,10000032,22595853,26862314,10000032-41,41.0,P23SJA,2180-05-07 01:00:00,2180-05-07 01:00:00,MAIN,Potassium Chloride,...,001275,2.450041e+08,10mEq ER Tablet,NaN,40,mEq,4,TAB,1.0,PO
4,10000032,22595853,30740602,10000032-27,27.0,P23SJA,2180-05-07 00:00:00,2180-05-07 22:00:00,MAIN,Sodium Chloride 0.9% Flush,...,NaN,0.000000e+00,10 mL Syringe,NaN,3,mL,0.3,SYR,3.0,IV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20292606,19999987,23865745,95605092,19999987-63,63.0,P213X2,2145-11-03 15:00:00,2145-11-03 18:00:00,MAIN,Propofol,...,65888.0,6.332303e+10,1000mg/100mL Vial,NaN,1000,mg,1,VIAL,0.0,IV DRIP
20292607,19999987,23865745,96309533,19999987-36,36.0,P59FJ8,2145-11-03 00:00:00,2145-11-04 13:00:00,BASE,0.9% Sodium Chloride,...,1210.0,3.380049e+08,100mL Bag,NaN,100,mL,100,mL,2.0,IV
20292608,19999987,23865745,96309533,19999987-36,36.0,P59FJ8,2145-11-03 00:00:00,2145-11-04 13:00:00,MAIN,LeVETiracetam,...,61005.0,1.478904e+10,500mg/5mL Vial,NaN,1000,mg,10,mL,2.0,IV
20292609,19999987,23865745,97298610,19999987-192,192.0,P47TKY,2145-11-08 16:00:00,2145-11-11 17:00:00,MAIN,Acetaminophen,...,4489.0,5.107900e+10,325mg Tablet,NaN,650,mg,2,TAB,NaN,PO/NG


In [14]:
prescriptions = prescriptions[prescriptions['drug_type'].isin(['MAIN'])]
prescriptions = prescriptions[['hadm_id', 'starttime', 'stoptime', 'drug']]
prescriptions = prescriptions.dropna(subset=['drug'])
valid_hadm_ids = ind_hosp['hadm_id'].unique()
prescriptions = prescriptions[prescriptions['hadm_id'].isin(valid_hadm_ids)]

In [15]:
prescriptions.isna().sum()

hadm_id          0
starttime    16524
stoptime     20894
drug             0
dtype: int64

In [16]:
prescriptions['starttime'] = pd.to_datetime(prescriptions['starttime'])
prescriptions['stoptime'] = pd.to_datetime(prescriptions['stoptime'], errors='coerce')
disch_times = ind_hosp[['hadm_id', 'dischtime']].drop_duplicates().set_index('hadm_id')
prescriptions = prescriptions.join(disch_times, on='hadm_id', how='left')

prescriptions['stoptime'] = prescriptions['stoptime'].fillna(prescriptions['dischtime'])
prescriptions = prescriptions[prescriptions['starttime'] <= prescriptions['stoptime']]
max_stoptime = prescriptions['starttime'] + pd.Timedelta(days=max_days)
prescriptions['stoptime'] = prescriptions['stoptime'].clip(upper=max_stoptime)

In [17]:
from collections import defaultdict
from tqdm import tqdm

meds_set = defaultdict(set)

for row in tqdm(prescriptions.itertuples(), total=len(prescriptions), desc="Processing prescriptions"):
    hadm_id = row.hadm_id
    start = row.starttime
    stop = row.stoptime
    drug = row.drug
    
    days_count = (stop - start).days + 1
    for i in range(days_count):
        current_date = (start + pd.Timedelta(days=i)).date()
        meds_set[(hadm_id, current_date)].add(drug)

meds_dict = {key: len(value) for key, value in meds_set.items()}
ind_hosp['meds_key'] = list(zip(ind_hosp['hadm_id'], ind_hosp['current_date']))
ind_hosp['num_medications_daily'] = ind_hosp['meds_key'].map(meds_dict).fillna(0).astype(int)
ind_hosp = ind_hosp.drop(columns=['meds_key'])

ind_hosp = ind_hosp.sort_values(['subject_id', 'hadm_id', 'day'])
ind_hosp['cumulative_medications'] = ind_hosp.groupby('hadm_id')['num_medications_daily'].cumsum()

print(f"Total days with prescriptions (unique drugs): {ind_hosp['num_medications_daily'].sum()}")
print(f"Max prescriptions per day (unique drugs): {ind_hosp['num_medications_daily'].max()}")

Processing prescriptions: 100%|██████████| 12995148/12995148 [03:36<00:00, 60074.27it/s]


Total days with prescriptions (unique drugs): 28513415
Max prescriptions per day (unique drugs): 60


### Labevents preprocessing (daily)

In [18]:
import pandas as pd

necessary_columns = [
    'hadm_id',
    'itemid',
    'valuenum',
    'charttime'
]

labevents = pd.read_csv('mimic-iv-3.1/hosp/labevents.csv.gz', usecols=necessary_columns)
labevents

,hadm_id,itemid,charttime,valuenum
0,NaN,50931,2180-03-23 11:51:00,95.00
1,NaN,51071,2180-03-23 11:51:00,NaN
2,NaN,51074,2180-03-23 11:51:00,NaN
3,NaN,51075,2180-03-23 11:51:00,NaN
4,NaN,51079,2180-03-23 11:51:00,NaN
...,...,...,...,...
158374759,23865745.0,51279,2145-11-09 05:30:00,3.52
158374760,23865745.0,51301,2145-11-09 05:30:00,5.70
158374761,NaN,50912,2146-02-07 11:13:00,1.10
158374762,NaN,50920,2146-02-07 11:13:00,NaN


In [19]:
labevents = labevents.dropna(subset=['hadm_id'])
labevents['charttime'] = pd.to_datetime(labevents['charttime'])
valid_hadm_ids = ind_hosp['hadm_id'].unique()
labevents = labevents[labevents['hadm_id'].isin(valid_hadm_ids)]

/var/folders/m0/9dr3jfm12bj1fkg2xjqh62jc0000gn/T/ipykernel_7037/3427018039.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  labevents['charttime'] = pd.to_datetime(labevents['charttime'])


In [20]:
key_itemids = [
    # Chemistry
    50912,   # Creatinine
    51006,   # BUN
    50931,   # Glucose
    50983,   # Sodium
    50971,   # Potassium
    50902,   # Chloride
    50882,   # Bicarbonate
    50893,   # Calcium (Total)
    50868,   # Anion Gap
    50960,   # Magnesium
    50970,   # Phosphate
    
    # Hematology
    51221,   # Hematocrit
    51222,   # Hemoglobin
    51301,   # WBC
    51265,   # Platelet Count
    51249,   # MCHC
    51279,   # Red Blood Cells
    51250,   # MCV
    51248,   # MCH
    51277    # RDW
]

labevents = labevents[labevents['itemid'].isin(key_itemids)]

In [21]:
labevents.isna().sum()

hadm_id          0
itemid           0
charttime        0
valuenum     51960
dtype: int64

In [22]:
labevents = labevents.dropna(subset=['valuenum'])

In [23]:
labevents = labevents.sort_values(['hadm_id', 'charttime'])

disch_times = ind_hosp[['hadm_id', 'dischtime']].drop_duplicates()
labevents = labevents.merge(disch_times, on='hadm_id', how='left')
labevents = labevents[labevents['charttime'] <= labevents['dischtime']]
labevents['chartdate'] = labevents['charttime'].dt.date

lab_last = labevents.groupby(['hadm_id', 'chartdate', 'itemid'])['valuenum'].last().reset_index()
lab_wide = lab_last.pivot_table(
    index=['hadm_id', 'chartdate'],
    columns='itemid',
    values='valuenum'
).reset_index()

lab_wide.columns = ['hadm_id', 'current_date'] + [f'lab_{itemid}_daily' for itemid in key_itemids]
ind_hosp = ind_hosp.merge(lab_wide, on=['hadm_id', 'current_date'], how='left')
for col in lab_wide.columns[2:]:
    ind_hosp[col] = ind_hosp[col].fillna(0)

In [24]:
# lab_features = labevents.groupby(['hadm_id', 'itemid'])['valuenum'].agg(
#     mean='mean',
#     min='min',
#     max='max', 
#     last='last',
#     count='count'
# ).reset_index()

# labevents['is_abnormal'] = (
#         (labevents['valuenum'] < labevents['ref_range_lower']) |
#         (labevents['valuenum'] > labevents['ref_range_upper'])
#     ).astype(int)
# abnormal = labevents.groupby(['hadm_id', 'itemid'])['is_abnormal'].mean().reset_index()
# abnormal.columns = ['hadm_id', 'itemid', 'abnormal_ratio']
# lab_features = lab_features.merge(abnormal, on=['hadm_id', 'itemid'], how='left')

# print(lab_features.head())

# lab_pivot = lab_features.pivot_table(
#     index='hadm_id',
#     columns='itemid',
#     values=['mean', 'min', 'max', 'last', 'count', 'abnormal_ratio']
# )

# lab_pivot.columns = [f'lab_{itemid}_{stat}' for stat, itemid in lab_pivot.columns]
# lab_pivot = lab_pivot.reset_index()

# ind_hosp = ind_hosp.merge(lab_pivot, on='hadm_id', how='left')
# lab_cols = [col for col in ind_hosp.columns if col.startswith('lab_')]
# ind_hosp[lab_cols] = ind_hosp[lab_cols].fillna(0)

### Imputing missing values

In [25]:
cols_with_na = ind_hosp.columns[ind_hosp.isna().sum() > 0].tolist()
print("Columns with missing values:", cols_with_na)

na_counts = ind_hosp[cols_with_na].isna().sum()
print(na_counts[na_counts > 0])

Columns with missing values: ['discharge_location', 'insurance', 'num_diagnoses', 'num_chronic', 'ccsr_FAC021', 'ccsr_FAC025', 'ccsr_END011', 'ccsr_CIR011', 'ccsr_END010', 'ccsr_CIR007', 'ccsr_END003', 'ccsr_CIR019', 'ccsr_DIG004', 'ccsr_CIR017', 'ccsr_CIR008', 'ccsr_BLD003', 'ccsr_EXT027', 'ccsr_GEN002', 'ccsr_END009', 'icd_I10', 'icd_E785', 'icd_K219', 'icd_Z87891', 'icd_I2510', 'icd_N179', 'icd_F329', 'icd_I4891', 'icd_Z7901', 'icd_F419', 'icd_E119', 'icd_E039', 'icd_Z794', 'icd_D649', 'icd_N390', 'comorbidity_score']
discharge_location    79737
insurance             19694
num_diagnoses           963
num_chronic             963
ccsr_FAC021             963
ccsr_FAC025             963
ccsr_END011             963
ccsr_CIR011             963
ccsr_END010             963
ccsr_CIR007             963
ccsr_END003             963
ccsr_CIR019             963
ccsr_DIG004             963
ccsr_CIR017             963
ccsr_CIR008             963
ccsr_BLD003             963
ccsr_EXT027             9

In [26]:
ind_hosp = ind_hosp[ind_hosp['discharge_location'] != 'DIED']
ind_hosp['discharge_location'] = ind_hosp['discharge_location'].fillna('UNKNOWN')
ind_hosp['insurance'] = ind_hosp['insurance'].fillna('UNKNOWN')
ind_hosp['insurance'] = ind_hosp['insurance'].replace({'No charge': 'Other'})

In [27]:
zero_fill_cols = [
    'num_diagnoses', 
    'num_chronic', 
    'comorbidity_score'
]
ccsr_cols = [col for col in ind_hosp.columns if col.startswith('ccsr_')]
zero_fill_cols.extend(ccsr_cols)

for col in zero_fill_cols:
    if col in ind_hosp.columns:
        ind_hosp[col] = ind_hosp[col].fillna(0).astype(int if col != 'comorbidity_score' else float)

### Target value (readmission in 30 days)

In [28]:
import pandas as pd

admissions_raw = pd.read_csv('mimic-iv-3.1/hosp/admissions.csv.gz')
admissions_raw['admittime'] = pd.to_datetime(admissions_raw['admittime'])
admissions_raw['dischtime'] = pd.to_datetime(admissions_raw['dischtime'])

admissions_raw = admissions_raw.sort_values(['subject_id', 'dischtime'])
admissions_raw['next_admittime'] = admissions_raw.groupby('subject_id')['admittime'].shift(-1)
admissions_raw['next_admission_type'] = admissions_raw.groupby('subject_id')['admission_type'].shift(-1)
planned_admission_types = ['ELECTIVE', 'SURGICAL SAME DAY ADMISSION']

next_admit_map = {}
for _, row in admissions_raw.iterrows():
    hadm_id = row['hadm_id']
    next_time = row['next_admittime']
    next_type = row['next_admission_type']
    
    if pd.notna(next_time) and next_type not in planned_admission_types:
        next_admit_map[hadm_id] = next_time
    else:
        next_admit_map[hadm_id] = None

ind_hosp['dischtime'] = pd.to_datetime(ind_hosp['dischtime']).dt.date

targets = []
for idx, row in ind_hosp.iterrows():
    next_date = next_admit_map.get(row['hadm_id'])
    if next_date is not None:
        current_date = pd.Timestamp(row['current_date'])
        days_diff = (next_date - current_date).days
        target = 1 if 0 < days_diff <= 30 else 0
    else:
        target = 0
    targets.append(target)

ind_hosp['target_readmission_30d'] = targets

In [29]:
ind_hosp

,subject_id,hadm_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,race,los,...,lab_51221_daily,lab_51222_daily,lab_51301_daily,lab_51265_daily,lab_51249_daily,lab_51279_daily,lab_51250_daily,lab_51248_daily,lab_51277_daily,target_readmission_30d
0,10000032,22841357,2180-06-26 18:27:00,2180-06-27,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,1
1,10000032,25742920,2180-08-05 23:44:00,2180-08-07,EW EMER.,EMERGENCY ROOM,HOSPICE,Medicaid,WHITE,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,1
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,2,...,34.8,11.9,34.9,34.1,102.0,94.0,15.8,3.40,4.1,1
4,10000084,23052089,2160-11-21 01:56:00,2160-11-25,EW EMER.,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,WHITE,4,...,39.8,13.3,31.9,33.4,95.0,286.0,12.9,4.17,8.1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1825053,19999987,23865745,2145-11-02 21:38:00,2145-11-11,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,...,37.5,12.3,34.3,32.7,105.0,135.0,15.9,3.58,10.0,0
1825054,19999987,23865745,2145-11-02 21:38:00,2145-11-11,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,...,38.4,12.7,35.1,33.2,106.0,141.0,15.9,3.63,5.9,0
1825055,19999987,23865745,2145-11-02 21:38:00,2145-11-11,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,...,35.1,11.5,34.5,32.7,106.0,129.0,15.8,3.32,5.0,0
1825056,19999987,23865745,2145-11-02 21:38:00,2145-11-11,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0


In [30]:
print(f"Readmission rate: {ind_hosp['target_readmission_30d'].mean():.2%}")
print(f"Distribution of target:\n{ind_hosp['target_readmission_30d'].value_counts()}")
print(f"Missing target: {ind_hosp['target_readmission_30d'].isna().sum()}")

Readmission rate: 20.07%
Distribution of target:
target_readmission_30d
0    1457616
1     366024
Name: count, dtype: int64
Missing target: 0


### Time-based features (prior_admissions_12mo)

In [31]:
admissions = pd.read_csv('mimic-iv-3.1/hosp/admissions.csv.gz')
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])

admissions_sorted = admissions.sort_values(['subject_id', 'admittime'])
admissions_sorted['prior_admissions_12mo'] = admissions_sorted.groupby('subject_id')['admittime'].transform(
    lambda x: [sum(1 for t in x.iloc[:i] if t >= x.iloc[i] - pd.Timedelta(days=365)) for i in range(len(x))]
)

In [32]:
ind_hosp = ind_hosp.merge(admissions_sorted[['hadm_id', 'prior_admissions_12mo']],
                          on='hadm_id', how='left')
ind_hosp

,subject_id,hadm_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,race,los,...,lab_51222_daily,lab_51301_daily,lab_51265_daily,lab_51249_daily,lab_51279_daily,lab_51250_daily,lab_51248_daily,lab_51277_daily,target_readmission_30d,prior_admissions_12mo
0,10000032,22841357,2180-06-26 18:27:00,2180-06-27,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,1,1
1,10000032,25742920,2180-08-05 23:44:00,2180-08-07,EW EMER.,EMERGENCY ROOM,HOSPICE,Medicaid,WHITE,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0,3
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,1,2
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,WHITE,2,...,11.9,34.9,34.1,102.0,94.0,15.8,3.40,4.1,1,2
4,10000084,23052089,2160-11-21 01:56:00,2160-11-25,EW EMER.,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,WHITE,4,...,13.3,31.9,33.4,95.0,286.0,12.9,4.17,8.1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1823635,19999987,23865745,2145-11-02 21:38:00,2145-11-11,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,...,12.3,34.3,32.7,105.0,135.0,15.9,3.58,10.0,0,0
1823636,19999987,23865745,2145-11-02 21:38:00,2145-11-11,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,...,12.7,35.1,33.2,106.0,141.0,15.9,3.63,5.9,0,0
1823637,19999987,23865745,2145-11-02 21:38:00,2145-11-11,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,...,11.5,34.5,32.7,106.0,129.0,15.8,3.32,5.0,0,0
1823638,19999987,23865745,2145-11-02 21:38:00,2145-11-11,EW EMER.,EMERGENCY ROOM,REHAB,Medicaid,UNKNOWN,8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0,0


### Categorical encoding

In [33]:
onehot_cols = [
    'admission_type',
    'discharge_location', 
    'insurance',
    'race',
    'admission_location'
]

binary_cols = ['gender']

ind_hosp = pd.get_dummies(
    ind_hosp,
    columns=onehot_cols,
    dummy_na=False,
    prefix=onehot_cols
)

ind_hosp['gender_male'] = (ind_hosp['gender'] == 'M').astype(int)
ind_hosp = ind_hosp.drop(columns=['gender', 'admittime'])
for col in onehot_cols:
    if col in ind_hosp.columns:
        ind_hosp = ind_hosp.drop(columns=[col])

In [ ]:
ind_hosp.to_parquet('ind_hosp.parquet', index=False)

: 